In [8]:
# Core imports shared by all examples
from __future__ import annotations

from typing import Any, List

from IPython.display import Image, display
from langgraph.graph import StateGraph, END , START
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from openinference.instrumentation.langchain import LangChainInstrumentor
from typing_extensions import Optional, Annotated, List, Sequence

from opentelemetry import trace
from langchain_core.tools import tool

import rich
from langgraph.graph.message import add_messages

import operator
from langchain_core.tools import tool, InjectedToolArg
import requests
import rich

from langchain_core.messages.ai import AIMessage

In [9]:
import dotenv
import os

In [10]:
dotenv.load_dotenv("../env_workshop")

True

In [11]:
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")


In [12]:
PHOENIX_PROJECT_NAME=os.environ.get("PHOENIX_PROJECT_NAME")

In [13]:
from phoenix.otel import register
from openinference.instrumentation import using_metadata

# configure the Phoenix tracer
tracer_provider = register(
  project_name=PHOENIX_PROJECT_NAME, 
  auto_instrument=False 
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

tracer = trace.get_tracer(__name__)


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: npatta01
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: llm-tracing.np-training.dev:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [14]:
from enum import StrEnum

# class syntax
class Operation(StrEnum):
    ADDITION = "add"
    MULTIPLICATION = "multiply"
    SUBTRACT = "subtract"
    DIVIDE = "divide"


In [15]:
@tool(parse_docstring=True)
def calculator(
    num_1: float,
    num_2: float,
    operation:Operation
) -> float:
    """Simple Calculator

    Args:
        num_1: number 1
        num_2: number 2
        operation: arithmetic operation to perform

    Returns:
        result of aritmetic operation
    """

    match operation:
        case Operation.ADDITION:
            return num_1 + num_2
        case Operation.SUBTRACT:
            return num_1 - num_2
        case Operation.DIVIDE:
            return num_1 / num_2
        case Operation.MULTIPLICATION:
            return num_1 * num_2
        case _:  # Default case, similar to 'else'
            ValueError("invalid operation")





In [16]:
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    openai_api_base=OPENAI_BASE_URL,
)

In [17]:
llm_with_tools = llm.bind_tools([calculator])

In [27]:
res = llm_with_tools.invoke( [ HumanMessage("what is 2 + 5") ] )
res

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 85, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CYtMzoFILLdKmNWGWN2VSOO0lq9JU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--6998c295-54d5-43b5-8018-74c40d5c12ad-0', tool_calls=[{'name': 'calculator', 'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'}, 'id': 'call_dDo7IySUjXR47HcEfsO3ikaJ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 85, 'output_tokens': 26, 'total_tokens': 111, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [28]:
res.tool_calls

[{'name': 'calculator',
  'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'},
  'id': 'call_dDo7IySUjXR47HcEfsO3ikaJ',
  'type': 'tool_call'}]

In [30]:
res = llm_with_tools.invoke( [ HumanMessage("what is 2 + 5 and then multiplied by 10"  ) ] )
res

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 91, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CYtNPGYDWStW29XoVNJuiUpzWjtuz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--145f894e-39ae-4426-9713-23425e82e4ba-0', tool_calls=[{'name': 'calculator', 'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'}, 'id': 'call_ThQSU8cJejiH8Xh2hXmORCbE', 'type': 'tool_call'}, {'name': 'calculator', 'args': {'num_1': 7, 'num_2': 10, 'operation': 'multiply'}, 'id': 'call_UuwufDG922ZVfwKpDAJgxcvv', 'type': 'tool_call'}], usage_metadata={'input_tokens': 91, 'output_tokens': 

In [31]:
res.tool_calls

[{'name': 'calculator',
  'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'},
  'id': 'call_ThQSU8cJejiH8Xh2hXmORCbE',
  'type': 'tool_call'},
 {'name': 'calculator',
  'args': {'num_1': 7, 'num_2': 10, 'operation': 'multiply'},
  'id': 'call_UuwufDG922ZVfwKpDAJgxcvv',
  'type': 'tool_call'}]

In [40]:
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage


In [41]:
class AgentState(BaseModel):
    messages: Annotated[List[BaseMessage], operator.add]


In [42]:
def call_model(state: AgentState):
    messages = state.messages
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

In [43]:
tool_node = ToolNode([calculator])


In [44]:
# Build the graph
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

# Set the entry point
workflow.add_edge(START, "agent")

# Add conditional edges from the agent node using the built-in condition
workflow.add_conditional_edges(
    "agent",
    tools_condition,
    # Map "tools" to the "tools" node, and the default ("__end__") to END
    {"tools": "tools", "__end__": END}
)

# Add a normal edge from the tools node back to the agent
workflow.add_edge("tools", "agent")

# Compile the graph
app = workflow.compile()

In [45]:
# Run the agent
result = app.invoke({"messages": [HumanMessage(content="what is 2 +5 ")]})

# Print the final response
print(result["messages"][-1].content)

The result of \(2 + 5\) is \(7\).


In [50]:
with tracer.start_as_current_span("tool_use_example"):
    result = app.invoke({"messages": [HumanMessage(content="what is 2 + 5" )]})

result["messages"]


[HumanMessage(content='what is 2 + 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 85, 'total_tokens': 111, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CYtMzoFILLdKmNWGWN2VSOO0lq9JU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--a9d9d5c5-1958-40fb-9937-85a9ce6f7e43-0', tool_calls=[{'name': 'calculator', 'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'}, 'id': 'call_dDo7IySUjXR47HcEfsO3ikaJ', 'type': 'tool_call'}], usage_metadata={'input_tokens': 85, 'output_tokens': 26, 'total_tokens': 111, 'input_token_details': {'audio': 0

In [47]:
result["messages"]

[HumanMessage(content='what is 2 + 5 and then multiplied by 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 91, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CYtNPGYDWStW29XoVNJuiUpzWjtuz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--545a2e6e-800a-400c-94e1-643b3c55939f-0', tool_calls=[{'name': 'calculator', 'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'}, 'id': 'call_ThQSU8cJejiH8Xh2hXmORCbE', 'type': 'tool_call'}, {'name': 'calculator', 'args': {'num_1': 7, 'num_2': 10, 'operation': 'multiply'}, 'id'

In [48]:
with tracer.start_as_current_span("tool_use_example"):
    result = app.invoke({"messages": [HumanMessage(content="what is 2 + 5 and then multiplied by 10" )]})

result["messages"]

[HumanMessage(content='what is 2 + 5 and then multiplied by 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 91, 'total_tokens': 159, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CYtNPGYDWStW29XoVNJuiUpzWjtuz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--b984b24c-02b7-47ca-b53d-89bfee9969f8-0', tool_calls=[{'name': 'calculator', 'args': {'num_1': 2, 'num_2': 5, 'operation': 'add'}, 'id': 'call_ThQSU8cJejiH8Xh2hXmORCbE', 'type': 'tool_call'}, {'name': 'calculator', 'args': {'num_1': 7, 'num_2': 10, 'operation': 'multiply'}, 'id'

### Tool call trace
![tool call trace](../images/tool_call_calculator.png)
